In [1]:
import os
import copy
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from tqdm import tqdm
from pytorch_fid import fid_score

In [2]:
#set random seed function
def set_seed(seed=242):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
#set random seed
set_seed(242)

In [3]:
#choose cpu/gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
#load and transform data
#transform part
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3)
])
#load part
dataset = datasets.CIFAR10(
    root='./data(CIFAR10)', train=False, download=True, transform=transform
)

  5%|▍         | 8.36M/170M [08:53<2:52:35, 15.7kB/s]


KeyboardInterrupt: 

In [20]:
#random select images
indices = torch.randperm(len(dataset))[:50]
images = torch.stack([dataset[i][0] for i in indices]).to(device)

In [21]:
#load U-Net.ipynb
%run U-Net.ipynb

In [22]:
#load Sigma-Sampler.ipynb
%run Sigma-Sampler.ipynb

In [24]:
#load model
model_baseline = UNet().to(device)
model_adaptive = UNet().to(device)
#load ckpt
ckpt_baseline = torch.load('checkpoint_baseline/step_60000_checkpoint.pth')
ckpt_adaptive = torch.load('checkpoint_adaptive/step_60000_checkpoint.pth')
model_baseline.load_state_dict(ckpt_baseline['model'])
model_adaptive.load_state_dict(ckpt_adaptive['model'])
#eval mode
model_baseline.eval()
model_adaptive.eval()
print('Checkpoint(baseline) Loaded Successfully...')
print('Checkpoint(adaptive) Loaded Successfully...')

Checkpoint(baseline) Loaded Successfully...
Checkpoint(adaptive) Loaded Successfully...


In [25]:
#generate images
with torch.no_grad():
    sigma_data = 0.5
    #sigma sampling
    sigma = sigma_sampler_lognormal(batch_size=images.shape[0]).to(device)
    sigma_reshaped = sigma.view(-1, 1, 1, 1)
    #generate noise
    noise = torch.randn_like(images)
    #calculate image with noise
    x_noisy = images + sigma_reshaped * noise
    #scaler factor - input image
    c_skip = sigma_data ** 2 / (sigma_reshaped ** 2 + sigma_data ** 2)
    c_out = sigma_reshaped * sigma_data / torch.sqrt(sigma_reshaped ** 2 + sigma_data ** 2)
    c_in = 1 / torch.sqrt(sigma_reshaped ** 2 + sigma_data ** 2)
    #calculate pred image
    pred_baseline = model_baseline(c_in * x_noisy, sigma)
    pred_adaptive = model_adaptive(c_in * x_noisy, sigma)
    #reconstruct image
    generated_baseline = c_skip * x_noisy + c_out * pred_baseline
    generated_adaptive = c_skip * x_noisy + c_out * pred_adaptive

#reverse normalization
#baseline
generated_baseline = generated_baseline.clamp(-1, 1)
generated_baseline = (generated_baseline + 1) / 2
generated_baseline = generated_baseline.clamp(0, 1)
#adaptive
generated_adaptive = generated_adaptive.clamp(-1, 1)
generated_adaptive = (generated_adaptive + 1) / 2
generated_adaptive = generated_adaptive.clamp(0, 1)
real_images = (images + 1) / 2

In [27]:
#save PNG
#baseline
save_dir = 'visual_samples'
os.makedirs(save_dir, exist_ok=True)

for i in range(50):
    #save real images
    save_image(
        real_images[i],
        os.path.join(save_dir, f'{i:03d}_real.png')
    )
    #save generated_baseline images
    save_image(
        generated_baseline[i],
        os.path.join(save_dir, f'{i:03d}_gen_base.png')
    )
    #save generated_adaptive images
    save_image(
        generated_adaptive[i],
        os.path.join(save_dir, f'{i:03d}_gen_adap.png')
    )